# Grid Search Occlusion Defence — Multilingual Attack

Defends against multilingual typographic attack using **4×4 grid occlusion**:
- **1-patch**: try all 16 patches, keep the occlusion with highest mean model confidence.
- **2-patch (greedy)**: fix best 1st patch, try remaining 15 for the 2nd best patch.

No GradCAM — purely black-box. Works by finding which patch, when removed, makes both models
most confident in their prediction.

Results saved to `results/grid_1patch/` and `results/grid_2patch/`.

In [1]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'open_clip_torch', 'transformers', 'datasets',
                'matplotlib', 'Pillow'], check=False)

CompletedProcess(args=['d:\\ian\\2026summer\\.venv\\Scripts\\python.exe', '-m', 'pip', 'install', '-q', 'open_clip_torch', 'transformers', 'datasets', 'matplotlib', 'Pillow'], returncode=0)

In [2]:
import importlib, sys, os, platform, random, json, time
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib import cm
import torch
import torch.nn.functional as F
import open_clip
from PIL import Image, ImageDraw, ImageFont, ImageFilter
from datasets import load_dataset
from transformers import ChineseCLIPModel, ChineseCLIPProcessor

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

LANGS = ['en', 'zh']
CLASSES = {
    'en': ['airplane', 'automobile', 'bird', 'cat', 'deer',
           'dog', 'frog', 'horse', 'ship', 'truck'],
    'zh': ['飞机', '汽车', '鸟', '猫', '鹿', '狗', '青蛙', '马', '船', '卡车'],
}
TMPL = {'en': 'a photo of a {}.', 'zh': '一张{}的照片。'}

RESULTS_DIR_1P = 'results/grid_1patch'
RESULTS_DIR_2P = 'results/grid_2patch'
os.makedirs(RESULTS_DIR_1P, exist_ok=True)
os.makedirs(RESULTS_DIR_2P, exist_ok=True)

SETUP_LABEL  = 'multilingual'
ATTACK_LABEL = 'multilingual'

Device: cuda


## 1. Model definitions

In [3]:
def classify(model, imgs, words, batch_size=128):
    preds = []
    for i in range(0, len(imgs), batch_size):
        imf = model.embed_images(imgs[i:i + batch_size])
        tf  = model.embed_texts(words)
        preds.append((imf @ tf.t()).argmax(-1).cpu().numpy())
    return np.concatenate(preds)

def classify_conf(model, imgs, words, batch_size=128):
    """Returns (predictions, max-cosine-sim confidences)."""
    preds, confs = [], []
    for i in range(0, len(imgs), batch_size):
        imf = model.embed_images(imgs[i:i + batch_size])
        tf  = model.embed_texts(words)
        sims = imf @ tf.t()
        preds.append(sims.argmax(-1).cpu().numpy())
        confs.append(sims.max(-1).values.cpu().numpy())
    return np.concatenate(preds), np.concatenate(confs)

def _clip_feat(out):
    if torch.is_tensor(out): return out
    if getattr(out, 'pooler_output', None) is not None: return out.pooler_output
    raise TypeError(type(out))

class EnCLIP:
    lang = 'en'
    def __init__(self):
        self.m, _, self.pp = open_clip.create_model_and_transforms('ViT-B-32', pretrained='openai')
        self.m = self.m.to(DEVICE).eval()
        self.tok = open_clip.get_tokenizer('ViT-B-32')
    @torch.no_grad()
    def embed_images(self, imgs):
        x = torch.stack([self.pp(im) for im in imgs]).to(DEVICE)
        return F.normalize(self.m.encode_image(x), dim=-1)
    @torch.no_grad()
    def embed_texts(self, words):
        t = self.tok([TMPL['en'].format(w) for w in words]).to(DEVICE)
        return F.normalize(self.m.encode_text(t), dim=-1)

class ZhCLIP:
    lang = 'zh'
    def __init__(self):
        self.m = ChineseCLIPModel.from_pretrained('OFA-Sys/chinese-clip-vit-base-patch16').to(DEVICE).eval()
        self.p = ChineseCLIPProcessor.from_pretrained('OFA-Sys/chinese-clip-vit-base-patch16')
    @torch.no_grad()
    def embed_images(self, imgs):
        pv = self.p(images=imgs, return_tensors='pt').pixel_values.to(DEVICE)
        return F.normalize(_clip_feat(self.m.get_image_features(pixel_values=pv)), dim=-1)
    @torch.no_grad()
    def embed_texts(self, words):
        t = self.p(text=[TMPL['zh'].format(w) for w in words], padding=True, return_tensors='pt').to(DEVICE)
        out = self.m.get_text_features(input_ids=t['input_ids'], attention_mask=t['attention_mask'],
                                        token_type_ids=t.get('token_type_ids'))
        return F.normalize(_clip_feat(out), dim=-1)

MODEL_CLS = {'en': EnCLIP, 'zh': ZhCLIP}
print('Model classes defined:', list(MODEL_CLS.keys()))

Model classes defined: ['en', 'zh']


## 2. Attack helpers

In [4]:
DISPLAY_SIZE = 224
NUM_BOXES    = 2
FONT_SIZE    = 24
PAD          = 8
_FONT_CACHE  = {}

def _font_paths():
    if platform.system() == 'Windows':
        winfonts = os.path.join(os.environ.get('WINDIR', r'C:\Windows'), 'Fonts')
        cjk   = os.path.join(winfonts, 'msyh.ttc')
        latin = os.path.join(winfonts, 'arial.ttf')
        if not os.path.exists(latin): latin = cjk
        return (cjk if os.path.exists(cjk) else None,
                latin if os.path.exists(latin) else None)
    for d in ['/usr/share/fonts', '/Library/Fonts', os.path.expanduser('~/.fonts')]:
        for f in ['NotoSansCJK-Regular.ttc', 'NotoSans-Regular.ttf']:
            p = os.path.join(d, f)
            if os.path.exists(p): return p, p
    return None, None

_CJK_FONT, _LAT_FONT = _font_paths()

def _font_for(word):
    return _CJK_FONT if any(ord(c) > 127 for c in word) else _LAT_FONT

def _get_font(fp, size=FONT_SIZE):
    key = (fp or '__default__', size)
    if key not in _FONT_CACHE:
        try:    _FONT_CACHE[key] = ImageFont.truetype(fp, size) if fp else ImageFont.load_default()
        except: _FONT_CACHE[key] = ImageFont.load_default()
    return _FONT_CACHE[key]

def _rects_overlap(a, b):
    return not (a[2] <= b[0] or b[2] <= a[0] or a[3] <= b[1] or b[3] <= a[1])

def _random_nonoverlapping_rect(rng, box_w, box_h, placed):
    x_hi = max(0, DISPLAY_SIZE - box_w)
    y_hi = max(0, DISPLAY_SIZE - box_h)
    rect_x, rect_y = 0, 0
    for _ in range(64):
        rect_x = rng.randint(0, x_hi) if x_hi > 0 else 0
        rect_y = rng.randint(0, y_hi) if y_hi > 0 else 0
        rect = (rect_x, rect_y, rect_x + box_w, rect_y + box_h)
        if all(not _rects_overlap(rect, p) for p in placed): return rect
    return (rect_x, rect_y, rect_x + box_w, rect_y + box_h)

def _draw_text_box(draw, word, rect, font):
    rx, ry, rx2, ry2 = rect
    bb = draw.textbbox((0, 0), word, font=font)
    draw.rectangle([rx, ry, rx2, ry2], fill='white')
    draw.text((rx + PAD - bb[0], ry + PAD - bb[1]), word, fill='black', font=font)

print('Font paths:', _CJK_FONT, _LAT_FONT)

Font paths: C:\WINDOWS\Fonts\msyh.ttc C:\WINDOWS\Fonts\arial.ttf


In [5]:
def draw_multilingual_attack(img, en_word, zh_word, img_idx, already_224=False):
    """Place English text at box-0 position, Chinese text at box-1 position."""
    if not already_224:
        img = img.convert('RGB').resize((DISPLAY_SIZE, DISPLAY_SIZE), Image.BICUBIC)
    else:
        img = img.copy()
    draw = ImageDraw.Draw(img)
    placed = []
    for box_i, (word, fp) in enumerate([(en_word, _LAT_FONT), (zh_word, _CJK_FONT)]):
        font = _get_font(fp)
        bb   = draw.textbbox((0, 0), word, font=font)
        bw   = (bb[2] - bb[0]) + 2 * PAD
        bh   = (bb[3] - bb[1]) + PAD + 12
        rng  = random.Random(int(img_idx) * NUM_BOXES + box_i)
        rect = _random_nonoverlapping_rect(rng, bw, bh, placed)
        placed.append(rect)
        _draw_text_box(draw, word, rect, font)
    return img

def build_multilingual_attacked_images(base_imgs, img_indices, n_workers=None):
    """Two boxes per image: EN attack word at box-0, ZH attack word at box-1."""
    n_workers = n_workers or min(8, os.cpu_count() or 4)
    tasks = [(im, int(k)) for im, k in zip(base_imgs, img_indices)]
    def _one(args):
        im, img_idx = args
        return draw_multilingual_attack(im,
                                        CLASSES['en'][target[img_idx]],
                                        CLASSES['zh'][target[img_idx]],
                                        img_idx, already_224=True)
    with ThreadPoolExecutor(max_workers=n_workers) as pool:
        return list(pool.map(_one, tasks))

print(f'Multilingual attack helper ready: {NUM_BOXES} boxes @ size {FONT_SIZE} (box-0=EN, box-1=ZH)')

Multilingual attack helper ready: 2 boxes @ size 24 (box-0=EN, box-1=ZH)


## 3. Dataset

In [6]:
hf = load_dataset('uoft-cs/cifar10', split='test')
label_key = 'label' if 'label' in hf.column_names else 'labels'
image_key = 'img'   if 'img'   in hf.column_names else 'image'

_sample_path = '../../image_samples/CIFAR10_BALANCED_1000_SAMPLE.json'
with open(_sample_path, encoding='utf-8') as f:
    _saved = json.load(f)

idx  = _saved['idx']
rows = hf.select(idx)
true = np.array(rows[label_key])
assert len(idx) == 1000
assert np.array_equal(true, np.array(_saved['true']))
assert all((true == c).sum() == 100 for c in range(10))

rng    = random.Random(0)
target = np.array([rng.choice([c for c in range(10) if c != int(true[k])])
                   for k in range(len(idx))])

clean     = [im.convert('RGB') for im in rows[image_key]]
print('Upscaling clean images to 224px...', end=' ', flush=True)
t0 = time.time()
clean_224 = [im.resize((DISPLAY_SIZE, DISPLAY_SIZE), Image.BICUBIC) for im in clean]
print(f'{time.time()-t0:.1f}s')

all_idx  = np.arange(len(clean))
tune_idx = np.concatenate([np.where(true == c)[0][:10] for c in range(10)])
print(f'Loaded {len(clean)} images; tune subset = {len(tune_idx)}')

Upscaling clean images to 224px... 0.2s
Loaded 1000 images; tune subset = 100


## 4. Load models + build attacked images

In [7]:
models = {}
for lang, cls in MODEL_CLS.items():
    t0 = time.time()
    print(f'Loading {lang}...', end=' ', flush=True)
    models[lang] = cls()
    print(f'{time.time()-t0:.1f}s')

# Standard text embeddings: each model with its own language
TEXT_EMB = {lang: models[lang].embed_texts(CLASSES[lang]).detach() for lang in LANGS}

clean_preds = {lang: classify(models[lang], clean_224, CLASSES[lang]) for lang in LANGS}
clean_acc   = {lang: float((clean_preds[lang] == true).mean()) for lang in LANGS}
print('Clean acc:', {k: f'{100*v:.1f}%' for k, v in clean_acc.items()})

Loading en... 

d:\ian\2026summer\.venv\Lib\site-packages\open_clip\factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


1.5s
Loading zh... 

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

1.6s
Clean acc: {'en': '85.9%', 'zh': '91.4%'}


In [8]:
print('Building attacked images...')
t0 = time.time()
attacked_imgs = build_multilingual_attacked_images(clean_224, all_idx)
print(f'Done in {time.time()-t0:.1f}s')

preds_attacked = {ml: classify(models[ml], attacked_imgs, CLASSES[ml]) for ml in LANGS}
baseline_acc   = {ml: float((preds_attacked[ml] == true).mean())   for ml in LANGS}
baseline_asr   = {ml: float((preds_attacked[ml] == target).mean()) for ml in LANGS}
print('Baseline acc:', {k: f'{100*v:.1f}%' for k, v in baseline_acc.items()})

Building attacked images...
Done in 0.2s
Baseline acc: {'en': '4.3%', 'zh': '7.3%'}


## 5. Grid defence helpers

**Scoring criterion**: after occluding a patch, take the maximum cosine-similarity (across 10
CIFAR classes) for both EN and ZH models and average them.  A higher score means both models
are more "confident" in some class prediction, which tends to happen when adversarial text
has been occluded.

In [9]:
GRID_ROWS = GRID_COLS = 4
PATCH_H   = PATCH_W   = DISPLAY_SIZE // GRID_ROWS   # 56 px
PATCHES   = [
    (c * PATCH_W, r * PATCH_H, (c + 1) * PATCH_W, (r + 1) * PATCH_H)
    for r in range(GRID_ROWS) for c in range(GRID_COLS)
]   # 16 non-overlapping patches

def occlude_rect(arr, rect, fill_color):
    out = arr.copy()
    x0, y0, x1, y1 = rect
    out[y0:y1, x0:x1] = fill_color
    return out

def score_candidates(candidates):
    """Return per-candidate score = mean max-cosine-sim across EN and ZH models."""
    scores = np.zeros(len(candidates))
    for ml in LANGS:
        _, confs = classify_conf(models[ml], candidates, CLASSES[ml])
        scores += confs
    return scores / len(LANGS)

print(f'Grid defence: {GRID_ROWS}x{GRID_COLS} grid = {len(PATCHES)} patches of {PATCH_H}x{PATCH_W}px')

Grid defence: 4x4 grid = 16 patches of 56x56px


In [10]:
def run_grid_1patch(imgs):
    """Find and apply the single best patch occlusion for each image."""
    result_imgs, best_patches = [], []
    t0 = time.time()
    for j, img in enumerate(imgs):
        arr        = np.array(img.convert('RGB'))
        fill_color = arr.reshape(-1, 3).mean(0).astype(np.uint8)
        candidates = [Image.fromarray(occlude_rect(arr, rect, fill_color)) for rect in PATCHES]
        scores     = score_candidates(candidates)
        best_pi    = int(scores.argmax())
        best_patches.append(best_pi)
        result_imgs.append(candidates[best_pi])
        if (j + 1) % 100 == 0:
            print(f'  Grid-1patch {j+1}/{len(imgs)} [{time.time()-t0:.1f}s]')
    return result_imgs, best_patches

def run_grid_2patch_greedy(imgs, first_patches):
    """Given 1st-patch results, greedily find the best 2nd patch."""
    result_imgs, second_patches = [], []
    t0 = time.time()
    for j, (img, fp) in enumerate(zip(imgs, first_patches)):
        arr        = np.array(img.convert('RGB'))
        fill_color = arr.reshape(-1, 3).mean(0).astype(np.uint8)
        arr1       = occlude_rect(arr, PATCHES[fp], fill_color)
        remain     = [i for i in range(len(PATCHES)) if i != fp]
        candidates = [Image.fromarray(occlude_rect(arr1, PATCHES[pi], fill_color)) for pi in remain]
        scores     = score_candidates(candidates)
        best_local = int(scores.argmax())
        second_pi  = remain[best_local]
        second_patches.append(second_pi)
        result_imgs.append(candidates[best_local])
        if (j + 1) % 100 == 0:
            print(f'  Grid-2patch {j+1}/{len(imgs)} [{time.time()-t0:.1f}s]')
    return result_imgs, second_patches

print('Grid defence run helpers ready.')

Grid defence run helpers ready.


## 6. Run 1-patch defence

In [11]:
print('Running 1-patch grid defence...')
t0 = time.time()
defended_1p, best_patches_1p = run_grid_1patch(attacked_imgs)
print(f'Done in {time.time()-t0:.1f}s')

preds_1p = {ml: classify(models[ml], defended_1p, CLASSES[ml]) for ml in LANGS}
acc_1p   = {ml: float((preds_1p[ml] == true).mean())   for ml in LANGS}
asr_1p   = {ml: float((preds_1p[ml] == target).mean()) for ml in LANGS}
print('1-patch defence acc:', {k: f'{100*v:.1f}%' for k, v in acc_1p.items()})
print('1-patch defence ASR:', {k: f'{100*v:.1f}%' for k, v in asr_1p.items()})

Running 1-patch grid defence...
  Grid-1patch 100/1000 [6.6s]
  Grid-1patch 200/1000 [12.8s]
  Grid-1patch 300/1000 [18.9s]
  Grid-1patch 400/1000 [25.1s]
  Grid-1patch 500/1000 [31.3s]
  Grid-1patch 600/1000 [37.5s]
  Grid-1patch 700/1000 [43.7s]
  Grid-1patch 800/1000 [49.8s]
  Grid-1patch 900/1000 [55.9s]
  Grid-1patch 1000/1000 [62.1s]
Done in 62.1s
1-patch defence acc: {'en': '5.4%', 'zh': '16.3%'}
1-patch defence ASR: {'en': '94.4%', 'zh': '83.3%'}


## 7. Run 2-patch defence (greedy)

In [12]:
print('Running 2-patch greedy grid defence...')
t0 = time.time()
defended_2p, second_patches = run_grid_2patch_greedy(attacked_imgs, best_patches_1p)
print(f'Done in {time.time()-t0:.1f}s')

preds_2p = {ml: classify(models[ml], defended_2p, CLASSES[ml]) for ml in LANGS}
acc_2p   = {ml: float((preds_2p[ml] == true).mean())   for ml in LANGS}
asr_2p   = {ml: float((preds_2p[ml] == target).mean()) for ml in LANGS}
print('2-patch defence acc:', {k: f'{100*v:.1f}%' for k, v in acc_2p.items()})
print('2-patch defence ASR:', {k: f'{100*v:.1f}%' for k, v in asr_2p.items()})

Running 2-patch greedy grid defence...
  Grid-2patch 100/1000 [5.9s]
  Grid-2patch 200/1000 [11.8s]
  Grid-2patch 300/1000 [17.7s]
  Grid-2patch 400/1000 [23.5s]
  Grid-2patch 500/1000 [29.4s]
  Grid-2patch 600/1000 [35.3s]
  Grid-2patch 700/1000 [41.3s]
  Grid-2patch 800/1000 [47.3s]
  Grid-2patch 900/1000 [53.2s]
  Grid-2patch 1000/1000 [59.0s]
Done in 59.0s
2-patch defence acc: {'en': '6.5%', 'zh': '16.9%'}
2-patch defence ASR: {'en': '93.2%', 'zh': '82.6%'}


## 8. Visual diagnostics

In [13]:
# Show which patch was selected for each class (use first 5 classes)
sample_pos = [int(np.where(true == c)[0][0]) for c in range(5)]
fig, axes  = plt.subplots(5, 4, figsize=(13, 17))
col_titles = ['Attacked', '1-patch occluded', '2-patch occluded', 'Patch grid']
for ax, t_title in zip(axes[0], col_titles):
    ax.set_title(t_title, fontsize=9, fontweight='bold')

for row_i, pos in enumerate(sample_pos):
    arr  = np.array(attacked_imgs[pos].convert('RGB'))
    fill = arr.reshape(-1, 3).mean(0).astype(np.uint8)

    # Grid overlay showing selected patches
    grid_img = attacked_imgs[pos].copy()
    grid_arr = np.array(grid_img)
    # Draw grid lines
    for r in range(1, GRID_ROWS):
        grid_arr[r * PATCH_H - 1:r * PATCH_H + 1, :] = [255, 0, 0]
    for c in range(1, GRID_COLS):
        grid_arr[:, c * PATCH_W - 1:c * PATCH_W + 1] = [255, 0, 0]
    # Highlight selected patches
    x0, y0, x1, y1 = PATCHES[best_patches_1p[pos]]
    grid_arr[y0:y0+3, x0:x1] = [0, 255, 0]
    grid_arr[y1-3:y1, x0:x1] = [0, 255, 0]
    grid_arr[y0:y1, x0:x0+3] = [0, 255, 0]
    grid_arr[y0:y1, x1-3:x1] = [0, 255, 0]
    x0, y0, x1, y1 = PATCHES[second_patches[pos]]
    grid_arr[y0:y0+3, x0:x1] = [0, 0, 255]
    grid_arr[y1-3:y1, x0:x1] = [0, 0, 255]
    grid_arr[y0:y1, x0:x0+3] = [0, 0, 255]
    grid_arr[y0:y1, x1-3:x1] = [0, 0, 255]

    panels = [attacked_imgs[pos], defended_1p[pos], defended_2p[pos], Image.fromarray(grid_arr)]
    for col_i, panel in enumerate(panels):
        axes[row_i, col_i].imshow(panel); axes[row_i, col_i].axis('off')
    axes[row_i, 0].set_ylabel(CLASSES['en'][true[pos]], fontsize=8, rotation=0, labelpad=36, va='center')

plt.suptitle(f'Grid defence examples — {SETUP_LABEL} attack', fontsize=11, y=1.001)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR_1P}/grid_defence_examples.png', dpi=120, bbox_inches='tight')
plt.close()
print('Saved -> grid_defence_examples.png')

Saved -> grid_defence_examples.png


## 9. Save results JSON

In [14]:
def save_grid_results(preds_def, acc_def, asr_def, method, cost, results_dir):
    defense_res = {}
    for ml in LANGS:
        base_p = preds_attacked[ml]
        new_p  = preds_def[ml]
        wrong  = base_p != true
        defense_res[ml] = {
            'acc':           float((new_p == true).mean()),
            'asr':           float((new_p == target).mean()),
            'recovery_rate': float((wrong & (new_p == true)).sum() / wrong.sum()) if wrong.any() else 0.0,
            'baseline_acc':  float((base_p == true).mean()),
            'baseline_asr':  float((base_p == target).mean()),
        }
    results = {
        'setup':            SETUP_LABEL,
        'method':           method,
        'attack':           ATTACK_LABEL,
        'n_images':         len(all_idx),
        'inference_cost':   cost,
        'clean_acc':        clean_acc,
        'baseline_acc':     {ml: defense_res[ml]['baseline_acc'] for ml in LANGS},
        'baseline_asr':     {ml: defense_res[ml]['baseline_asr'] for ml in LANGS},
        'defense':          defense_res,
        'defense_acc_mean': float(np.mean([defense_res[ml]['acc'] for ml in LANGS])),
        'defense_asr_mean': float(np.mean([defense_res[ml]['asr'] for ml in LANGS])),
    }
    out_path = f'{results_dir}/confusion_results_grid.json'
    with open(out_path, 'w', encoding='utf-8') as f:
        json.dump(results, f, indent=2, ensure_ascii=False)
    print(f'Saved -> {out_path}')
    return results

res_1p = save_grid_results(preds_1p, acc_1p, asr_1p, 'grid_1patch', 32, RESULTS_DIR_1P)
res_2p = save_grid_results(preds_2p, acc_2p, asr_2p, 'grid_2patch', 62, RESULTS_DIR_2P)

print('\n=== Grid Defence Summary ===')
for label, res in [('1-patch', res_1p), ('2-patch', res_2p)]:
    print(f'{label}:')
    for ml in LANGS:
        d = res['defense'][ml]
        print(f'  model_{ml}: {100*d["baseline_acc"]:.1f}% -> {100*d["acc"]:.1f}%  '
              f'ASR {100*d["baseline_asr"]:.1f}% -> {100*d["asr"]:.1f}%')

Saved -> results/grid_1patch/confusion_results_grid.json
Saved -> results/grid_2patch/confusion_results_grid.json

=== Grid Defence Summary ===
1-patch:
  model_en: 4.3% -> 5.4%  ASR 95.5% -> 94.4%
  model_zh: 7.3% -> 16.3%  ASR 92.7% -> 83.3%
2-patch:
  model_en: 4.3% -> 6.5%  ASR 95.5% -> 93.2%
  model_zh: 7.3% -> 16.9%  ASR 92.7% -> 82.6%
